# Topological Deep Learning Challenge 2026: Bridging the Gap

This notebook evaluates your model across a comprehensive grid of **GraphUniverse** synthetic graph distributions. GraphUniverse ([Van Langendonck et al., ICLR 2026](https://openreview.net/forum?id=jRWxvQnqUt)) is a framework for generating controlled synthetic graph families with specific structural properties, enabling rigorous evaluation of inductive generalization capabilities.

## Dataset Generation

Graphs are generated using a stochastic block model with community-aware edge probabilities. The grid spans three key structural axes:
- **Homophily** (low [0.0-0.1], mid [0.4-0.6], high [0.9-1.0]): Controls the tendency of nodes to connect within vs. across communities
- **Average degree** (low [1-2], high [4-5]): Controls graph sparsity
- **Power-law exponent** (low [1.5-2.0], high [4.0-5.0]): Controls degree distribution heterogeneity

This creates a 3×2×2 = 12 distinct distribution settings. Each setting generates 100 graphs with 50-300 nodes and 5-10 unique community types per graph drawn from a total of 20 unique latent communities.

## Tasks

**1. Community Detection (node-level, inductive)**  
Predict the latent community label for each node. Communities are the ground-truth structural clusters used during graph generation. This tests whether models can recover the underlying community structure from observed edges.

**2. Triangle Counting (graph-level, inductive)**  
Predict the total number of triangles (3-cliques) in each graph. This regression task evaluates the model's ability to capture higher-order structural patterns beyond pairwise edges.

## Evaluation Protocol

- Each of the 12 grid settings is trained with **3 random seeds** (36 total training runs per task)
- The **best checkpoint** from each run is evaluated on:
  - **In-distribution test set**: Same distribution as training
  - **Out-of-distribution (OOD) test sets**: All other 11 grid settings
- This produces a comprehensive analysis of how structural properties (homophily, degree, etc.) affect generalization

**OOD evaluation** reveals whether models overfit to specific structural regimes or learn robust representations that transfer across diverse graph topologies.

## Setup Requirements

**Make sure Weights & Biases is configured:**
```bash
wandb login
```

## How to Use

1. Set your `MODEL_CONFIG` (path to your model yaml)
2. Run the evaluation cells below
3. **Submit the generated `results.json` file with your PR** — this file contains all metrics used for challenge evaluation
4. Review the generated heatmap visualizations to understand your model's performance profile

**IMPORTANT:** DO NOT modify this notebook or `utils.py`. This may invalidate your submission.

## Results

After running the evaluation, you will find in the output directory:
- **`results.json`** (REQUIRED FOR SUBMISSION): Contains all training metrics, in-distribution test scores, and OOD evaluation results for every grid setting and seed.
- **Heatmap visualizations**: PNG files showing mean ± std performance across the 12 grid settings (in-distribution)
- **OOD delta plots**: PNG files in the `OOD/` subdirectory showing performance change (OOD − in-distribution) organized by homophily level

In [ ]:
import sys
from pathlib import Path

# Setup paths
_ROOT = Path.cwd().resolve()
_REPO = _ROOT if (_ROOT / "configs" / "run.yaml").exists() else _ROOT.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

from utils import (
    resolve_project_root,
    run_challenge_grid,
    check_challenge_grid,
    save_challenge_artifacts,
)

PROJECT_ROOT = resolve_project_root(_REPO)

## Configuration

**Set your model config path here:**

In [ ]:
# Your model configuration (e.g., "graph/gcn", "graph/gin", "graph/gat")
MODEL_CONFIG = "graph/gin"

## Sanity check

This check ensures that no notebook cells have been modified below this point. Most submissions should only change the model MODEL_CONFIG variable above.

In [ ]:
expected_hash = "f87b2cfd0c61cabaa50ee2c05d8fa7b5e1d78e14e8c3a50dfdd933a5b389bfde"

In [ ]:
# UNIQUE_HASH_MARKER
import json
import hashlib

def hash_remaining_cells(notebook_path: str, marker_string: str = "# UNIQUE_HASH_MARKER") -> str:
    """
    Computes a SHA-256 hash of all cells from the marker cell to the end of the notebook.

    Args:
        notebook_path (str): The hardcoded path to the notebook file.
        marker_string (str, optional): A unique string present in the source of the calling cell.
                                       Defaults to "# UNIQUE_HASH_MARKER".

    Returns:
        str: The SHA-256 hash of the content of the target cells.

    Raises:
        ValueError: If the marker_string is not found in the notebook data.
    """
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook_data = json.load(f)

    cells = notebook_data.get('cells', [])
    start_index = -1

    for i, cell in enumerate(cells):
        source = "".join(cell.get('source', []))
        if marker_string in source[:len(marker_string)]:
            start_index = i
            break

    if start_index == -1:
        raise ValueError(f"Marker string '{marker_string}' not found in the notebook.")

    content_to_hash = ""
    for cell in cells[start_index:]:
        content_to_hash += "".join(cell.get('source', []))

    return hashlib.sha256(content_to_hash.encode('utf-8')).hexdigest()

hash_value = hash_remaining_cells(notebook_path="run_evaluation.ipynb")

print(f"Computed hash: {hash_value}")

if hash_value != expected_hash:
    raise ValueError("❌ The content of the notebook has been modified. Please ensure that you are using the original notebook without any changes from this point onward.")
else:
    print("✅ Notebook content is verified.")


## Check runs

Here we quickly check that all configurations run correctly before running the full evaluation. The first time this will also create all the needed datasets.

In [ ]:
check_challenge_grid(
    project_root=PROJECT_ROOT,
    model_config=MODEL_CONFIG,
    quiet=False,
)

## Run Evaluation

This will train and evaluate your model across all grid configurations.

In [ ]:
results, study_id = run_challenge_grid(
    project_root=PROJECT_ROOT,
    model_config=MODEL_CONFIG,
)

## Save Results

Generate JSON output and visualization plots.

In [ ]:
output_paths = save_challenge_artifacts(
    results,
    model_config=MODEL_CONFIG,
    study_id=study_id,
)

print(f"\nResults saved to: {output_paths['dir']}")
print(f"JSON: {output_paths['json']}")

## Inspect Results

View results as a DataFrame for quick inspection.

In [ ]:
import pandas as pd

pd.DataFrame(results)